# Multilayer Perceptrons
You should build an end-to-end machine learning pipeline using a multilayer perceptron model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [multilayer perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [3]:
import pandas as pd

In [4]:
url = "https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv"
df= pd.read_csv(url)
df.head(10)

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,61207,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,33799,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,5414,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,61377,6,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1875,9,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
X = df.drop(columns=['id','class'])
y = df['class']

In [6]:
#split train rest
from sklearn.model_selection import train_test_split
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=42)


In [7]:
# Build pipeline: StandardScaler + MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),   # Feature scaling (important for MLP!)
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(128, 64),  # Two hidden layers
        activation='relu',             # ReLU activation
        solver='adam',                 # Adam optimizer
        learning_rate_init=0.001,      # Learning rate
        max_iter=50,                   # Max epochs
        early_stopping=True,           # Stop when validation doesn't improve
        validation_fraction=0.1,       # 10% of training for validation
        n_iter_no_change=10,           # Patience for early stopping
        random_state=42,
        verbose=False
    ))
])

print('Pipeline created!')
print(pipeline)

Pipeline created!
Pipeline(steps=[('scaler', StandardScaler()),
                ('mlp',
                 MLPClassifier(early_stopping=True,
                               hidden_layer_sizes=(128, 64), max_iter=50,
                               random_state=42))])


In [10]:
#train model
pipeline.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()),
                ('mlp',
                 MLPClassifier(early_stopping=True,
                               hidden_layer_sizes=(128, 64), max_iter=50,
                               random_state=42))])

In [11]:
# Check training info
mlp = pipeline.named_steps['mlp']

In [13]:
#optimizing pipeline
# GridSearchCV to find best hyperparameters
from sklearn.model_selection import GridSearchCV
param_grid = {
    'mlp__hidden_layer_sizes': [(64,), (128, 64), (256, 128)],
    'mlp__learning_rate_init': [0.001, 0.01],
    'mlp__alpha': [0.0001, 0.001],  # L2 regularization
}
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,           # 3-fold cross validation
    scoring='accuracy',
    n_jobs=-1,      # Use all CPU cores
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('mlp',
                                        MLPClassifier(early_stopping=True,
                                                      hidden_layer_sizes=(128,
                                                                          64),
                                                      max_iter=50,
                                                      random_state=42))]),
             n_jobs=-1,
             param_grid={'mlp__alpha': [0.0001, 0.001],
                         'mlp__hidden_layer_sizes': [(64,), (128, 64),
                                                     (256, 128)],
                         'mlp__learning_rate_init': [0.001, 0.01]},
             scoring='accuracy', verbose=1)

In [14]:
# Use best model
best_model = grid_search.best_estimator_

In [19]:
#evaluate
# Predict on test set
from sklearn.metrics import accuracy_score
y_pred = best_model.predict(X_rest)

# Accuracy
accuracy = accuracy_score(y_rest, y_pred)
print(f'Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

Test Accuracy: 0.9081 (90.81%)
